In [1]:
# distributed storage,
# distributed computation,
# fault tolerance,
# parallel execution.

# data does not fit comfortably in memory,
# transformations are expensive,
# many files must be processed,
# pipelines must scale.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

In [3]:
spark = SparkSession.builder.appName('analysis').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/17 21:52:31 WARN Utils: Your hostname, Thomaskuttys-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.5 instead (on interface en0)
26/05/17 21:52:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 21:52:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/17 21:52:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/17 21:52:32 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


### Loading data csv

In [4]:
df = spark.read.csv('transactions.csv',header = True)

In [5]:
df.show(5,truncate=False)

+--------------+-----------+-------+-----------+-------+----------+--------+--------+----------+--------+--------------+
|transaction_id|customer_id|city   |category   |product|txn_date  |txn_time|quantity|unit_price|discount|payment_method|
+--------------+-----------+-------+-----------+-------+----------+--------+--------+----------+--------+--------------+
|T1            |C101       |Chennai|Electronics|Laptop |2024-01-01|09:10:15|1       |65000     |0.10    |UPI           |
|T2            |C102       |Mumbai |Electronics|Phone  |2024-01-01|10:05:33|2       |20000     |0.05    |Card          |
|T3            |C103       |Chennai|Furniture  |Chair  |2024-01-01|11:20:10|4       |1500      |0.00    |Cash          |
|T4            |C101       |Chennai|Electronics|Mouse  |2024-01-01|12:15:45|2       |800       |0.20    |UPI           |
|T5            |C104       |Delhi  |Appliances |Mixer  |2024-01-01|13:40:21|1       |3000      |0.10    |Card          |
+--------------+-----------+----

### Creating New column : Coverting string to time stamp

In [6]:
df = (
    df
    .withColumn(
        'txn_timestamp',
        F.to_timestamp(
            F.concat_ws(' ', F.col('txn_date'), F.col('txn_time')),
            "yyyy-MM-dd HH:mm:ss"
        )
    )
    .withColumn(
        'revenue',
        F.col('quantity') * (F.col('unit_price') * (1 - F.col('discount').cast(T.DoubleType())))
    )
    .withColumn('txn_hour',F.hour('txn_timestamp'))
    # .withColumn('txn_day',F.day('txn_timestamp')) # we only have 5 day txn
    # .withColumn('txn_month',F.month('txn_timestamp')) # we only have 1 month data
)


### Selecting columns from dataframe

In [7]:
df.select('txn_date','txn_time','txn_timestamp','txn_hour','revenue').show(5,truncate=False)

+----------+--------+-------------------+--------+-------+
|txn_date  |txn_time|txn_timestamp      |txn_hour|revenue|
+----------+--------+-------------------+--------+-------+
|2024-01-01|09:10:15|2024-01-01 09:10:15|9       |58500.0|
|2024-01-01|10:05:33|2024-01-01 10:05:33|10      |38000.0|
|2024-01-01|11:20:10|2024-01-01 11:20:10|11      |6000.0 |
|2024-01-01|12:15:45|2024-01-01 12:15:45|12      |1280.0 |
|2024-01-01|13:40:21|2024-01-01 13:40:21|13      |2700.0 |
+----------+--------+-------------------+--------+-------+
only showing top 5 rows


### OrderBy and select

In [8]:
df.orderBy(F.desc(F.col('txn_date'))).select('txn_date','transaction_id').show(truncate=False)
# another method :  df.orderBy(F.col('txn_date').desc()).show(5)

+----------+--------------+
|txn_date  |transaction_id|
+----------+--------------+
|2024-01-05|T42           |
|2024-01-05|T41           |
|2024-01-05|T43           |
|2024-01-05|T44           |
|2024-01-05|T45           |
|2024-01-05|T46           |
|2024-01-05|T47           |
|2024-01-05|T48           |
|2024-01-05|T49           |
|2024-01-05|T50           |
|2024-01-04|T31           |
|2024-01-04|T32           |
|2024-01-04|T33           |
|2024-01-04|T34           |
|2024-01-04|T35           |
|2024-01-04|T36           |
|2024-01-04|T37           |
|2024-01-04|T38           |
|2024-01-04|T39           |
|2024-01-04|T40           |
+----------+--------------+
only showing top 20 rows


### Distinct elements

In [9]:
df.select(F.col('payment_method')).distinct().show()

+--------------+
|payment_method|
+--------------+
|          Card|
|          Cash|
|           UPI|
+--------------+



In [10]:
df.agg(
    F.count_distinct('category'),
    F.count_distinct('city'),
    F.count_distinct('product'),
    F.count_distinct('payment_method')
).show()

+------------------------+--------------------+-----------------------+------------------------------+
|count(DISTINCT category)|count(DISTINCT city)|count(DISTINCT product)|count(DISTINCT payment_method)|
+------------------------+--------------------+-----------------------+------------------------------+
|                       3|                   4|                     11|                             3|
+------------------------+--------------------+-----------------------+------------------------------+



In [11]:
df.select(F.col('city')).distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|  Chennai|
|   Mumbai|
|    Delhi|
+---------+



In [12]:
def get_distinct_counts(dataframe, target_columns):
    """
    getting the number of unique values in the target_columns as a dataframe
    """
    # 1. Dynamically build the count_distinct expressions for each column
    # We construct a string literal for the column name and pair it with the count
    array_elements = [
        F.struct(F.lit(col).alias("column_name"), F.count_distinct(col).alias("distinct_count"))
        for col in target_columns
    ]

    result_df = (
        dataframe
        .agg(
            F.array(*array_elements).alias("counts_array")
        )
        .select(F.explode("counts_array").alias("result"))
        .select("result.column_name", "result.distinct_count")
    )

    return result_df

categorical_cols = ['category', 'city', 'product', 'payment_method']
distinct_summary_df = get_distinct_counts(df, categorical_cols)

In [13]:
distinct_summary_df.show()

+--------------+--------------+
|   column_name|distinct_count|
+--------------+--------------+
|      category|             3|
|          city|             4|
|       product|            11|
|payment_method|             3|
+--------------+--------------+



### Basic Groupby and Pivot

In [14]:
(
    df
    .groupBy('city')
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('customer_id').alias('unique_customers_count'),
        F.max('revenue').alias('highest_single_transaction'),
        F.avg(F.col('discount').cast(T.DoubleType())).alias('avg_deal_discount')
    )
).show()

+---------+-------------+----------------------+--------------------------+-------------------+
|     city|total_revenue|unique_customers_count|highest_single_transaction|  avg_deal_discount|
+---------+-------------+----------------------+--------------------------+-------------------+
|Bangalore|      60680.0|                     6|                   23400.0|0.05833333333333334|
|  Chennai|     325325.0|                    20|                   64600.0|0.09000000000000004|
|   Mumbai|     337560.0|                    12|                   64800.0|0.08750000000000001|
|    Delhi|     189750.0|                    12|                   63750.0|0.07916666666666666|
+---------+-------------+----------------------+--------------------------+-------------------+



In [15]:
df.groupBy('city').agg(
    F.sum('revenue').alias('total_revenue'),
    F.round(F.mean('revenue'),2).alias('mean_revenue'),
    F.min('revenue').alias('min_revenue')
).show()

+---------+-------------+------------+-----------+
|     city|total_revenue|mean_revenue|min_revenue|
+---------+-------------+------------+-----------+
|Bangalore|      60680.0|    10113.33|     2280.0|
|  Chennai|     325325.0|    16266.25|     1045.0|
|   Mumbai|     337560.0|     28130.0|     3600.0|
|    Delhi|     189750.0|     15812.5|     2185.0|
+---------+-------------+------------+-----------+



In [16]:
df.groupBy('city').pivot('payment_method').sum('revenue').show()

+---------+--------+--------+--------+
|     city|    Card|    Cash|     UPI|
+---------+--------+--------+--------+
|Bangalore|    NULL| 60680.0|    NULL|
|  Chennai| 93660.0|  6000.0|225665.0|
|   Mumbai|212750.0|    NULL|124810.0|
|    Delhi|  7165.0|157900.0| 24685.0|
+---------+--------+--------+--------+



In [17]:
# multiple groups and then pivot 
df.groupBy('city','category').pivot('payment_method').sum('revenue').show()

+---------+-----------+--------+--------+--------+
|     city|   category|    Card|    Cash|     UPI|
+---------+-----------+--------+--------+--------+
|  Chennai|Electronics| 93660.0|    NULL|154580.0|
|Bangalore|  Furniture|    NULL| 16000.0|    NULL|
|  Chennai|  Furniture|    NULL|  6000.0| 54400.0|
|    Delhi| Appliances|  7165.0|    NULL| 24685.0|
|Bangalore|Electronics|    NULL| 44680.0|    NULL|
|    Delhi|Electronics|    NULL|157900.0|    NULL|
|   Mumbai|Electronics|212750.0|    NULL| 59500.0|
|   Mumbai|  Furniture|    NULL|    NULL| 65310.0|
|  Chennai| Appliances|    NULL|    NULL| 16685.0|
+---------+-----------+--------+--------+--------+



In [18]:
### Statistical summary for multiple columns

In [19]:
df.select('revenue','unit_price').summary().show()

+-------+-----------------+----------------+
|summary|          revenue|      unit_price|
+-------+-----------------+----------------+
|  count|               50|              50|
|   mean|          18266.3|         18500.0|
| stddev|20082.22312196556|22765.3738141494|
|    min|           1045.0|            1100|
|    25%|           3060.0|          1500.0|
|    50%|           7790.0|          8000.0|
|    75%|          22950.0|         25000.0|
|    max|          64800.0|             950|
+-------+-----------------+----------------+



In [20]:
df.groupBy('city').pivot('payment_method').sum('revenue').show()

+---------+--------+--------+--------+
|     city|    Card|    Cash|     UPI|
+---------+--------+--------+--------+
|Bangalore|    NULL| 60680.0|    NULL|
|  Chennai| 93660.0|  6000.0|225665.0|
|   Mumbai|212750.0|    NULL|124810.0|
|    Delhi|  7165.0|157900.0| 24685.0|
+---------+--------+--------+--------+



In [21]:
df.groupBy('category').pivot('payment_method').agg(
    F.sum(
        F.col('quantity').cast(T.IntegerType())
    )
).show()

+-----------+----+----+---+
|   category|Card|Cash|UPI|
+-----------+----+----+---+
|Electronics|  18|  11|  9|
|  Furniture|NULL|  10| 20|
| Appliances|   5|NULL| 11|
+-----------+----+----+---+



### Filtering the data 

In [22]:
df_bangalore = df.filter(F.col('city')=='Bangalore')

In [23]:
df_bangalore.select('transaction_id','city','category','product').show()

+--------------+---------+-----------+-------+
|transaction_id|     city|   category|product|
+--------------+---------+-----------+-------+
|            T7|Bangalore|  Furniture|  Table|
|           T17|Bangalore|  Furniture|  Table|
|           T25|Bangalore|Electronics| Tablet|
|           T33|Bangalore|  Furniture|  Chair|
|           T41|Bangalore|Electronics|  Phone|
|           T49|Bangalore|Electronics|  Mouse|
+--------------+---------+-----------+-------+



In [24]:
df.filter(
    (F.col('city')=='Bangalore')&
    (F.col('category')=='Electronics')
).select(
    'transaction_id','city','category'
).show()

+--------------+---------+-----------+
|transaction_id|     city|   category|
+--------------+---------+-----------+
|           T25|Bangalore|Electronics|
|           T41|Bangalore|Electronics|
|           T49|Bangalore|Electronics|
+--------------+---------+-----------+



In [25]:
target_cities = ['Mumbai','Bangalore']
df.filter(F.col('City').isin(target_cities)).select('transaction_id','city','category').show()

+--------------+---------+-----------+
|transaction_id|     city|   category|
+--------------+---------+-----------+
|            T2|   Mumbai|Electronics|
|            T6|   Mumbai|Electronics|
|            T7|Bangalore|  Furniture|
|           T11|   Mumbai|Electronics|
|           T14|   Mumbai|  Furniture|
|           T17|Bangalore|  Furniture|
|           T19|   Mumbai|Electronics|
|           T22|   Mumbai|  Furniture|
|           T25|Bangalore|Electronics|
|           T27|   Mumbai|Electronics|
|           T30|   Mumbai|  Furniture|
|           T33|Bangalore|  Furniture|
|           T35|   Mumbai|Electronics|
|           T38|   Mumbai|  Furniture|
|           T41|Bangalore|Electronics|
|           T43|   Mumbai|Electronics|
|           T46|   Mumbai|  Furniture|
|           T49|Bangalore|Electronics|
+--------------+---------+-----------+



In [26]:
# .contains(), .startswith() , .endswith()
df.filter(F.col('product').contains('Phone')).select('transaction_id','product').show()

+--------------+-------+
|transaction_id|product|
+--------------+-------+
|            T2|  Phone|
|            T8|  Phone|
|           T13|  Phone|
|           T21|  Phone|
|           T27|  Phone|
|           T31|  Phone|
|           T41|  Phone|
|           T45|  Phone|
+--------------+-------+



In [27]:
df_product_counts = df.groupBy('product').count()
df_product_counts.show()

+----------+-----+
|   product|count|
+----------+-----+
|     Chair|    5|
|     Phone|    8|
|    Laptop|    7|
| Microwave|    3|
|      Sofa|    4|
|     Table|    4|
|     Mouse|    4|
|   Toaster|    4|
|     Mixer|    5|
|    Tablet|    4|
|Headphones|    2|
+----------+-----+



In [28]:
total_rows = df.count()

In [29]:
total_rows

50

In [30]:
df_product_counts = df_product_counts.withColumn('ratio',F.col('count')/total_rows)

In [31]:
df_product_counts.show()

+----------+-----+-----+
|   product|count|ratio|
+----------+-----+-----+
|     Chair|    5|  0.1|
|     Phone|    8| 0.16|
|    Laptop|    7| 0.14|
| Microwave|    3| 0.06|
|      Sofa|    4| 0.08|
|     Table|    4| 0.08|
|     Mouse|    4| 0.08|
|   Toaster|    4| 0.08|
|     Mixer|    5|  0.1|
|    Tablet|    4| 0.08|
|Headphones|    2| 0.04|
+----------+-----+-----+



In [32]:
df_product_counts.agg(F.sum('ratio')).show()

+------------------+
|        sum(ratio)|
+------------------+
|0.9999999999999999|
+------------------+



### Window functions

In [33]:
city_window = Window.partitionBy('city').orderBy(F.asc('txn_timestamp'))

In [34]:
df = (
    df
    .withColumn('lag_revenue', F.lag('revenue', 1).over(city_window))
    .withColumn('running_total', F.sum('revenue').over(city_window)) # Added this column
)

In [35]:
df.filter(F.col('city')=='Bangalore').orderBy(F.asc('txn_timestamp')).select('city','txn_timestamp','revenue','running_total','lag_revenue').show()

+---------+-------------------+-------+-------------+-----------+
|     city|      txn_timestamp|revenue|running_total|lag_revenue|
+---------+-------------------+-------+-------------+-----------+
|Bangalore|2024-01-01 15:30:40| 5000.0|       5000.0|       NULL|
|Bangalore|2024-01-02 15:10:05| 5200.0|      10200.0|     5000.0|
|Bangalore|2024-01-03 13:30:10|23400.0|      33600.0|     5200.0|
|Bangalore|2024-01-04 11:30:25| 5800.0|      39400.0|    23400.0|
|Bangalore|2024-01-05 09:05:10|19000.0|      58400.0|     5800.0|
|Bangalore|2024-01-05 17:25:30| 2280.0|      60680.0|    19000.0|
+---------+-------------------+-------+-------------+-----------+



In [36]:
df.columns

['transaction_id',
 'customer_id',
 'city',
 'category',
 'product',
 'txn_date',
 'txn_time',
 'quantity',
 'unit_price',
 'discount',
 'payment_method',
 'txn_timestamp',
 'revenue',
 'txn_hour',
 'lag_revenue',
 'running_total']

In [37]:
customer_window = Window.partitionBy('customer_id').orderBy('txn_timestamp')
df = (
    df
    .withColumn('purchase_rank',F.row_number().over(customer_window))
    .withColumn('cum_quantity',F.sum(F.col('quantity')).over(customer_window))
    .withColumn('prev_timestamp',F.lag('txn_timestamp',1).over(customer_window))
    .withColumn('days_since_last_purchase',F.datediff(F.col('txn_timestamp'),F.col('prev_timestamp')))
)

In [38]:
df.select('customer_id','txn_timestamp','prev_timestamp','days_since_last_purchase','purchase_rank','quantity','cum_quantity').show()

+-----------+-------------------+-------------------+------------------------+-------------+--------+------------+
|customer_id|      txn_timestamp|     prev_timestamp|days_since_last_purchase|purchase_rank|quantity|cum_quantity|
+-----------+-------------------+-------------------+------------------------+-------------+--------+------------+
|       C101|2024-01-01 09:10:15|               NULL|                    NULL|            1|       1|         1.0|
|       C101|2024-01-01 12:15:45|2024-01-01 09:10:15|                       0|            2|       2|         3.0|
|       C101|2024-01-01 18:45:00|2024-01-01 12:15:45|                       0|            3|       1|         4.0|
|       C101|2024-01-02 16:40:30|2024-01-01 18:45:00|                       1|            4|       2|         6.0|
|       C101|2024-01-03 14:45:00|2024-01-02 16:40:30|                       1|            5|       2|         8.0|
|       C101|2024-01-04 12:40:35|2024-01-03 14:45:00|                       1|  

### Exercise : 
 * Identifying premium buyers in electronics sales
 * Calculate their purchase momentum
 * how they choose to pay across different cities 

In [39]:
df = spark.read.csv('transactions.csv',header = True)

In [40]:
df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- txn_date: string (nullable = true)
 |-- txn_time: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- payment_method: string (nullable = true)



In [41]:
pipeline_1 = (
    df
    .withColumn(
        'txn_timestamp',
        F.to_timestamp(F.concat_ws(' ', F.col('txn_date'), F.col('txn_time')), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn('revenue',F.col('quantity').cast(T.IntegerType())*F.col('unit_price')*(1-F.col('discount').cast(T.DoubleType())))
    .filter(F.col('category') == 'Electronics')
    .withColumn(
        "revenue_segment", 
        F.when(F.col("revenue").cast(T.DoubleType()) >= 50000, "Premium")  # Fixed to >=
         .otherwise("Standard")
    )
)
customer_window = Window.partitionBy('customer_id').orderBy('txn_timestamp')

pipeline_2 = (
    pipeline_1
    .withColumn('running_electronics_revenue',F.sum('revenue').over(customer_window))
)

pipeline_3 = (
    pipeline_2
    .groupBy('city','revenue_segment')
    .pivot('payment_method')
    .agg(F.count_distinct('customer_id'))
    .orderBy('city')
)
              

In [42]:
pipeline_3.show()

+---------+---------------+----+----+----+
|     city|revenue_segment|Card|Cash| UPI|
+---------+---------------+----+----+----+
|Bangalore|       Standard|NULL|   1|NULL|
|  Chennai|       Standard|   1|NULL|   2|
|  Chennai|        Premium|   1|NULL|   1|
|    Delhi|        Premium|NULL|   1|NULL|
|    Delhi|       Standard|NULL|   1|NULL|
|   Mumbai|        Premium|   1|NULL|   1|
|   Mumbai|       Standard|   2|NULL|NULL|
+---------+---------------+----+----+----+



### Missing values : 

In [43]:
df.agg(
    F.sum(F.when(F.col('quantity').isNull() | F.isnan('quantity'),1).otherwise(0)).alias('missing values')
).show()

+--------------+
|missing values|
+--------------+
|             0|
+--------------+



In [54]:
def get_missing_value_counts(dataframe):
    array_elements = [
        F.struct(F.lit(col).alias('column_name'),F.sum(F.when(F.col(col).isNull(),1).otherwise(0)).alias('missing_count'))
        for col in dataframe.columns
    ]
    result_df = (
        dataframe
        .agg(
            F.array(*array_elements).alias('missing_count_array')
        )
        .select(F.explode('missing_count_array').alias('result'))
        .select('result.column_name','result.missing_count')        
    )
    return result_df
    
get_missing_value_counts(df).show()

+--------------+-------------+
|   column_name|missing_count|
+--------------+-------------+
|transaction_id|            0|
|   customer_id|            0|
|          city|            0|
|      category|            0|
|       product|            0|
|      txn_date|            0|
|      txn_time|            0|
|      quantity|            0|
|    unit_price|            0|
|      discount|            0|
|payment_method|            0|
+--------------+-------------+



In [47]:
# create sample data with missing values 
schema = T.StructType([
    T.StructField('transaction_id',T.StringType(),True),
    T.StructField('customer_name',T.StringType(),True),
    T.StructField('city',T.StringType(),True),
    T.StructField('amount',T.DoubleType(),True),
    T.StructField('discount',T.FloatType(),True)
])
schema

StructType([StructField('transaction_id', StringType(), True), StructField('customer_name', StringType(), True), StructField('city', StringType(), True), StructField('amount', DoubleType(), True), StructField('discount', FloatType(), True)])

In [57]:
sample_data = [
    ("T1", "Alice", "Chennai", 5000.0, 0.1),
    ("T2", "Bob", None, None, 0.05),            # Missing City (Null String)
    ("T3", None, "Mumbai", None, 0.0),             # Missing Name & Amount
    ("T4", "Charlie", "Delhi", 3500.0, float('nan')), # NaN Discount (Math NaN)
    (None, "David", "Bangalore", 8000.0, 0.15)     # Missing ID
]

In [58]:
d_df = spark.createDataFrame(sample_data,schema = schema)

In [59]:
d_df.show()

+--------------+-------------+---------+------+--------+
|transaction_id|customer_name|     city|amount|discount|
+--------------+-------------+---------+------+--------+
|            T1|        Alice|  Chennai|5000.0|     0.1|
|            T2|          Bob|     NULL|  NULL|    0.05|
|            T3|         NULL|   Mumbai|  NULL|     0.0|
|            T4|      Charlie|    Delhi|3500.0|     NaN|
|          NULL|        David|Bangalore|8000.0|    0.15|
+--------------+-------------+---------+------+--------+



In [56]:
get_missing_value_counts(d_df).show()

+--------------+-------------+
|   column_name|missing_count|
+--------------+-------------+
|transaction_id|            1|
| customer_name|            1|
|          city|            1|
|        amount|            1|
|      discount|            0|
+--------------+-------------+



In [68]:
### replacing a specific value - returning the new dataframe : pyspark dataframes are immutable 

In [64]:
d_df = (
    d_df
    .withColumn('row_index',F.monotonically_increasing_id())
)

In [65]:
d_df.show()

+--------------+-------------+---------+------+--------+-----------+
|transaction_id|customer_name|     city|amount|discount|  row_index|
+--------------+-------------+---------+------+--------+-----------+
|            T1|        Alice|  Chennai|5000.0|     0.1| 8589934592|
|            T2|          Bob|     NULL|  NULL|    0.05|25769803776|
|            T3|         NULL|   Mumbai|  NULL|     0.0|42949672960|
|            T4|      Charlie|    Delhi|3500.0|     NaN|60129542144|
|          NULL|        David|Bangalore|8000.0|    0.15|77309411328|
+--------------+-------------+---------+------+--------+-----------+



In [67]:
(
    d_df
    .withColumn(
        'city',
        F.when(F.col('row_index')==25769803776,F.lit('Bangalore')).otherwise(F.col('city'))
    )
).show()

+--------------+-------------+---------+------+--------+-----------+
|transaction_id|customer_name|     city|amount|discount|  row_index|
+--------------+-------------+---------+------+--------+-----------+
|            T1|        Alice|  Chennai|5000.0|     0.1| 8589934592|
|            T2|          Bob|Bangalore|  NULL|    0.05|25769803776|
|            T3|         NULL|   Mumbai|  NULL|     0.0|42949672960|
|            T4|      Charlie|    Delhi|3500.0|     NaN|60129542144|
|          NULL|        David|Bangalore|8000.0|    0.15|77309411328|
+--------------+-------------+---------+------+--------+-----------+



In [70]:
clean_df = (
    d_df
    .na.fill(
        {
            'transaction_id':'unknown',
            'customer_name':'guest',
            'city':'others',
            'amount':0.0,
            'discount':0.0
        }
    )
)

In [72]:
clean_df.show()

+--------------+-------------+---------+------+--------+-----------+
|transaction_id|customer_name|     city|amount|discount|  row_index|
+--------------+-------------+---------+------+--------+-----------+
|            T1|        Alice|  Chennai|5000.0|     0.1| 8589934592|
|            T2|          Bob|   others|   0.0|    0.05|25769803776|
|            T3|        guest|   Mumbai|   0.0|     0.0|42949672960|
|            T4|      Charlie|    Delhi|3500.0|     0.0|60129542144|
|       unknown|        David|Bangalore|8000.0|    0.15|77309411328|
+--------------+-------------+---------+------+--------+-----------+



In [73]:
# text_cleaned_df = dirty_df.na.fill("Missing Text", subset=["city", "customer_name"])

### Joins  - Left join

In [74]:
# Your Transaction DataFrame: df (contains customer_id, revenue, etc.)

# Create a secondary Customer Profiles lookup table
customer_profiles = spark.createDataFrame([
    ("C101", "Gold Member", "2022-04-12"),
    ("C102", "Silver Member", "2023-01-15"),
    ("C103", "Bronze Member", "2024-02-01"),
    ("C199", "VIP Member", "2021-11-11") # Note: C199 has no transactions in your data
], ["customer_id", "loyalty_tier", "signup_date"])


In [75]:
customer_profiles.show()

+-----------+-------------+-----------+
|customer_id| loyalty_tier|signup_date|
+-----------+-------------+-----------+
|       C101|  Gold Member| 2022-04-12|
|       C102|Silver Member| 2023-01-15|
|       C103|Bronze Member| 2024-02-01|
|       C199|   VIP Member| 2021-11-11|
+-----------+-------------+-----------+



In [76]:
customer_profiles.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: string (nullable = true)



In [77]:
left_joined_df = df.join(customer_profiles,on='customer_id',how='left')

In [78]:
left_joined_df.count()

50

In [85]:
left_joined_df.select('customer_id','loyalty_tier','signup_date').show()

+-----------+-------------+-----------+
|customer_id| loyalty_tier|signup_date|
+-----------+-------------+-----------+
|       C104|         NULL|       NULL|
|       C104|         NULL|       NULL|
|       C104|         NULL|       NULL|
|       C107|         NULL|       NULL|
|       C107|         NULL|       NULL|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C103|Bronze Member| 2024-02-01|
|       C103|Bronze Member| 2024-02-01|
|       C103|Bronze Member| 2024-02-01|
|       C105|         NULL|       NULL|
|       C105|         NULL|       NULL|
|       C108|         NULL|       NULL|
|       C108|         NULL|       NULL|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C106|         NULL|       NULL|
|       C101|  Gold Member| 2022-04-12|
+-----------+-------------+-----------+
only showing top 20 rows


In [86]:
df.groupBy('customer_id').agg(F.count('customer_id')).show()

+-----------+------------------+
|customer_id|count(customer_id)|
+-----------+------------------+
|       C104|                 7|
|       C107|                 5|
|       C102|                 7|
|       C103|                 7|
|       C105|                 6|
|       C108|                 5|
|       C101|                 8|
|       C106|                 5|
+-----------+------------------+



### Joins - Inner

In [87]:
inner_joined_df = df.join(customer_profiles,on='customer_id',how='inner')
inner_joined_df.select('customer_id','loyalty_tier','signup_date').show()

+-----------+-------------+-----------+
|customer_id| loyalty_tier|signup_date|
+-----------+-------------+-----------+
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C101|  Gold Member| 2022-04-12|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C102|Silver Member| 2023-01-15|
|       C103|Bronze Member| 2024-02-01|
|       C103|Bronze Member| 2024-02-01|
|       C103|Bronze Member| 2024-02-01|
|       C103|Bronze Member| 2024-02-01|
|       C103|Bronze Member| 2024-02-01|
+-----------+-------------+-----------+
only showing top 20 rows


## Exercise  - Joins + Null value replacement 

In [89]:

# join transaction with the customr profiles and fill the newly added
# loyalty_tier with 'Standard Guest' value 

In [101]:
output_df = (
    df
    .join(customer_profiles,on='customer_id',how='left')
    .withColumn('loyalty_tier',F.when(F.col('loyalty_tier').isNull(),'Standard Guest').otherwise(F.col('loyalty_tier')))
)

output_df = (
    df
    .join(customer_profiles, on='customer_id', how='left')
    .na.fill({'loyalty_tier': 'Standard Guest'}) # Shorter alternative
)


In [102]:
output_df.select('customer_id','loyalty_tier','signup_date').show(1000)

+-----------+--------------+-----------+
|customer_id|  loyalty_tier|signup_date|
+-----------+--------------+-----------+
|       C104|Standard Guest|       NULL|
|       C104|Standard Guest|       NULL|
|       C104|Standard Guest|       NULL|
|       C104|Standard Guest|       NULL|
|       C104|Standard Guest|       NULL|
|       C104|Standard Guest|       NULL|
|       C104|Standard Guest|       NULL|
|       C107|Standard Guest|       NULL|
|       C107|Standard Guest|       NULL|
|       C107|Standard Guest|       NULL|
|       C107|Standard Guest|       NULL|
|       C107|Standard Guest|       NULL|
|       C102| Silver Member| 2023-01-15|
|       C102| Silver Member| 2023-01-15|
|       C102| Silver Member| 2023-01-15|
|       C102| Silver Member| 2023-01-15|
|       C102| Silver Member| 2023-01-15|
|       C102| Silver Member| 2023-01-15|
|       C102| Silver Member| 2023-01-15|
|       C103| Bronze Member| 2024-02-01|
|       C103| Bronze Member| 2024-02-01|
|       C103| Br

## Joins with Different Column names

In [103]:
# Regional marketing tier lookup table
marketing_regions = spark.createDataFrame([
    ("Chennai", "Tier 1 Hub"),
    ("Mumbai", "Tier 1 Hub"),
    ("Delhi", "Tier 1 Hub"),
    ("Bangalore", "Tier 2 Hub")
], ["location_name", "market_segment"])

In [105]:
marketing_regions.show()

+-------------+--------------+
|location_name|market_segment|
+-------------+--------------+
|      Chennai|    Tier 1 Hub|
|       Mumbai|    Tier 1 Hub|
|        Delhi|    Tier 1 Hub|
|    Bangalore|    Tier 2 Hub|
+-------------+--------------+



In [110]:
output = (
    df
    .join(marketing_regions,on = df['city']==marketing_regions['location_name'],how='left')
)

output.select('transaction_id','city','location_name','market_segment').show(1000)

+--------------+---------+-------------+--------------+
|transaction_id|     city|location_name|market_segment|
+--------------+---------+-------------+--------------+
|            T7|Bangalore|    Bangalore|    Tier 2 Hub|
|           T17|Bangalore|    Bangalore|    Tier 2 Hub|
|           T25|Bangalore|    Bangalore|    Tier 2 Hub|
|           T33|Bangalore|    Bangalore|    Tier 2 Hub|
|           T41|Bangalore|    Bangalore|    Tier 2 Hub|
|           T49|Bangalore|    Bangalore|    Tier 2 Hub|
|            T1|  Chennai|      Chennai|    Tier 1 Hub|
|            T3|  Chennai|      Chennai|    Tier 1 Hub|
|            T4|  Chennai|      Chennai|    Tier 1 Hub|
|            T8|  Chennai|      Chennai|    Tier 1 Hub|
|           T10|  Chennai|      Chennai|    Tier 1 Hub|
|           T12|  Chennai|      Chennai|    Tier 1 Hub|
|           T15|  Chennai|      Chennai|    Tier 1 Hub|
|           T18|  Chennai|      Chennai|    Tier 1 Hub|
|           T20|  Chennai|      Chennai|    Tier

### Saving the data into disk 

In [111]:
# Partition the data physically by City
# df.write.partitionBy("city").mode("overwrite").parquet("output/partitioned_sales")
# Writing as a high-performance Parquet file
# df.write.mode("overwrite").parquet("output/optimized_transactions")
# df.write.mode("overwrite").option("header", "true").csv("output/clean_transactions")
